### 1 - Attach libraries

In [1]:
import pandas as pd
import numpy as np
import requests
import glob
import os

### 2 - Functions to read data

In [42]:
def cargar_predicciones_por_partido(ruta_archivos):
    """
    Carga los archivos parquet y los estructura a nivel de partido.
    """
    archivos = glob.glob(f"{ruta_archivos}/pred*.parquet")
    df_pred = pd.concat(
        [pd.read_parquet(archivo).assign(File=os.path.basename(archivo)) for archivo in archivos], 
        ignore_index=True
    )
    
    # Nos quedamos con las columnas clave, ¡incluyendo la nueva que acabamos de crear!
    df_pred = df_pred[['Date', 'Team', 'Opponent2', 'Goles_Estimados', 'File']]
    
    # Como asumo que tienes una fila por equipo (ej: Fila 1-> Chile vs Perú, Fila 2-> Perú vs Chile),
    # vamos a crear una clave única por partido ordenando alfabéticamente los nombres para evitar duplicados.
    df_pred['Partido_ID'] = df_pred.apply(
        lambda x: "-".join(sorted([x['Team'], x['Opponent2']])), axis=1
    )
    
    # Separamos en Equipo A y Equipo B para tener todo en una sola fila por partido
    df_partidos = df_pred.drop_duplicates(subset=['Partido_ID', 'File']).copy()
    df_partidos = df_partidos.rename(columns={'Team': 'Team_A', 'Opponent2': 'Team_B', 'Goles_Estimados': 'Pred_Goles_A'})
    
    # Buscamos la predicción del Equipo B en el dataframe original
    pred_b = df_pred.rename(columns={'Team': 'Team_B', 'Opponent2': 'Team_A', 'Goles_Estimados': 'Pred_Goles_B'})
    
    # Unimos ambas predicciones en la misma fila
    df_final_pred = pd.merge(df_partidos, pred_b[['Team_B', 'Pred_Goles_B', 'Partido_ID', 'File']], 
                on=['File','Partido_ID','Team_B'], how='left')    
    
    return df_final_pred

In [46]:
def obtener_resultados_reales(url_games, url_teams):
    """
    Consume las APIs de partidos y equipos, y hace un join para obtener los nombres en inglés.
    """
    # 1. Obtener la data de las APIs
    res_games = requests.get(url_games)
    res_teams = requests.get(url_teams)
    
    if res_games.status_code != 200 or res_teams.status_code != 200:
        raise Exception("Error al conectar con una de las APIs.")
        
    # Extraer los datos (asumiendo que la lista viene en la raíz o en una llave 'data')
    data_games = res_games.json()
    data_teams = res_teams.json()
    
    # Si el JSON tiene una raíz 'data', extraemos la lista. Si es una lista directa, la pasamos tal cual.
    lista_games = data_games.get('games', data_games) if isinstance(data_games, dict) else data_games
    lista_teams = data_teams.get('teams', data_teams) if isinstance(data_teams, dict) else data_teams

    # 2. Crear los DataFrames
    df_games = pd.DataFrame(lista_games)
    df_teams = pd.DataFrame(lista_teams)[['id', 'fifa_code']].rename(columns={'id':'id_team'}) # Solo nos interesa el ID y el nombre
    
    # 3. Filtrar solo los partidos que ya terminaron (opcional, pero recomendado)
    if 'finished' in df_games.columns:
        df_games = df_games[df_games['finished'].astype(str).str.upper() == "TRUE"]

    # 4. Cruce para el Equipo Local (Home)
    df_games = pd.merge(df_games, df_teams, left_on='home_team_id', right_on='id_team', how='left')
    df_games = df_games.rename(columns={'fifa_code': 'Team_A_real'})
    df_games = df_games.drop(columns=['id_team']) # Eliminamos el 'id' de teams para que no estorbe en el siguiente cruce
    
    # 5. Cruce para el Equipo Visitante (Away)
    df_games = pd.merge(df_games, df_teams, left_on='away_team_id', right_on='id_team', how='left')
    df_games = df_games.rename(columns={'fifa_code': 'Team_B_real'})
    df_games = df_games.drop(columns=['id_team']) # Eliminamos el 'id' de teams para que no estorbe en el siguiente cruce

    # 6. Formatear y preparar las columnas finales
    # Forzamos conversión numérica por si la API devuelve los goles como string ("2", "0")
    df_games['Goles_A'] = pd.to_numeric(df_games['home_score'], errors='coerce')
    df_games['Goles_B'] = pd.to_numeric(df_games['away_score'], errors='coerce')
    
    # Creamos la misma llave única de partido (ordenada alfabéticamente)
    df_games['Partido_ID'] = df_games.apply(
        lambda x: "-".join(sorted([str(x['Team_A_real']), str(x['Team_B_real'])])), axis=1
    )
    
    return df_games[['Partido_ID','Team_A_real','Team_B_real','Goles_A','Goles_B']]

In [52]:
def signo_resultado(goles_a, goles_b):
    """
    Devuelve 1 si gana A, -1 si gana B, 0 si empatan.
    """
    if pd.isna(goles_a) or pd.isna(goles_b):
        return np.nan
    if goles_a > goles_b: return 1
    elif goles_a < goles_b: return -1
    else: return 0

### 3 - Read and preprocess data

In [73]:
# 1. Cargar datos
ruta_parquet = '../database'  # Ruta donde tienes guardados los archivos
url_games = "https://worldcup26.ir/get/games"
url_teams = "https://worldcup26.ir/get/teams" # URL de la API de equipos

print("Cargando predicciones de Parquet...")
df_pred = cargar_predicciones_por_partido(ruta_parquet)

df_pred['Date'] = pd.to_datetime(df_pred['Date'])
condicion_eliminar = ((df_pred['File'] == 'predictions_f2.parquet') & (df_pred['Date'] > '2026-06-23'))
df_pred = df_pred[~condicion_eliminar]

print("Consumiendo API y cruzando resultados reales...")
df_real = obtener_resultados_reales(url_games, url_teams)

# 2. Unir predicciones con resultados reales
df = pd.merge(df_pred, df_real, on='Partido_ID', how='inner')
df['Real_Goles_A'] = np.where(df['Team_A']==df['Team_A_real'], df['Goles_A'], df['Goles_B'])
df['Real_Goles_B'] = np.where(df['Team_B']==df['Team_B_real'], df['Goles_B'], df['Goles_A'])

# 3. Calcular las 3 métricas
df['Acierto_Goles_A'] = df['Pred_Goles_A'] == df['Real_Goles_A']
df['Acierto_Goles_B'] = df['Pred_Goles_B'] == df['Real_Goles_B']

total_equipos_evaluados = len(df) * 2
aciertos_goles_exactos = df['Acierto_Goles_A'].sum() + df['Acierto_Goles_B'].sum()
met_1_goles_exactos = (aciertos_goles_exactos / total_equipos_evaluados) * 100

df['Tendencia_Pred'] = df.apply(lambda x: signo_resultado(x['Pred_Goles_A'], x['Pred_Goles_B']), axis=1)
df['Tendencia_Real'] = df.apply(lambda x: signo_resultado(x['Real_Goles_A'], x['Real_Goles_B']), axis=1)

df['Acierto_Ganador'] = df['Tendencia_Pred'] == df['Tendencia_Real']
met_2_ganador = df['Acierto_Ganador'].mean() * 100

df['Acierto_Marcador_Exacto'] = df['Acierto_Goles_A'] & df['Acierto_Goles_B']
met_3_resultado_exacto = df['Acierto_Marcador_Exacto'].mean() * 100

Cargando predicciones de Parquet...
Consumiendo API y cruzando resultados reales...


In [75]:
# 4. Mostrar Resultados
print("\n" + "="*40)
print("MÉTRICAS DE DESEMPEÑO DEL MODELO")
print("="*40)
print(f"Total de partidos evaluados: {len(df)}")
print(f"1. Goles Exactos (por equipo):   {met_1_goles_exactos:.2f}% de acierto")
print(f"2. Ganador / Empate correcto:    {met_2_ganador:.2f}% de acierto")
print(f"3. Marcador Exacto del partido:  {met_3_resultado_exacto:.2f}% de acierto")
print("="*40)


MÉTRICAS DE DESEMPEÑO DEL MODELO
Total de partidos evaluados: 64
1. Goles Exactos (por equipo):   33.59% de acierto
2. Ganador / Empate correcto:    65.62% de acierto
3. Marcador Exacto del partido:  10.94% de acierto
